**Outline - Data loading & quick preview**
- Import `pandas` and `numpy`.
- Load `processed_dataset.csv` from the data folder.
- Optional sampling via `SAMPLE_SIZE` for faster runs.
- Print dataset size and show the first rows.

In [30]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/processed_dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

SAMPLE_SIZE = None 
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print("Menggunakan sampel:", len(df))
else:
    print("Menggunakan semua data.")

print("Jumlah baris:", len(df))
print("Jumlah kolom:", df.shape[1])
display(df.head())

Menggunakan semua data.
Jumlah baris: 99994
Jumlah kolom: 80


,derived_msa_md,state_code,county_code,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,preapproval,lien_status,reverse_mortgage,loan_amount,...,loan_purpose_32,loan_purpose_4,loan_purpose_5,income_bracket_High,income_bracket_Low,income_bracket_Medium,income_bracket_Very High,loan_size_Large,loan_size_Medium,loan_size_Small
0,12580,md,24005.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,465000,...,0,0,0,0,0,0,1,1,0,0
1,22220,ar,5007.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,125000,...,0,0,0,0,0,0,1,0,0,1
2,43900,sc,45083.0,c,conventional:subordinate lien,single family (1-4 units):site-built,2,2,2,55000,...,0,1,0,0,0,0,0,0,0,1
3,19430,oh,39057.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,65000,...,0,0,0,0,1,0,0,0,0,1
4,41700,tx,48029.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,125000,...,0,0,0,0,0,0,0,0,0,1


**Outline - Item/feature preparation**
- Define helper to cap high-cardinality categories.
- Build binary columns from prefix rules and force 0/1 format.
- Clean categorical columns (missing/sentinel values) and one-hot encode.
- Bin numeric columns into quantiles, then one-hot encode.
- Merge all items, drop redundant/missing-only items, and sanity-check counts.

In [31]:
def cap_categories(series, top_n=10, other_label="other"):
    value_counts = series.value_counts(dropna=True)
    top = value_counts.nlargest(top_n).index
    return series.where(series.isin(top), other_label)

binary_prefixes = [
    "derived_sex_",
    "derived_race_",
    "derived_ethnicity_",
    "loan_type_",
    "loan_purpose_",
    "purchaser_type_",
    "income_bracket_",
    "loan_size_",
]

binary_cols = [c for c in df.columns if any(c.startswith(p) for p in binary_prefixes)]
binary_df = df[binary_cols].copy()

binary_df = binary_df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
binary_df = binary_df.clip(0, 1)

cat_cols = [
    "state_code",
    "conforming_loan_limit",
    "derived_loan_product_type",
    "derived_dwelling_category",
    "preapproval",
    "lien_status",
    "reverse_mortgage",
    "occupancy_type",
    "construction_method",
    "total_units",
    "applicant_age",
    "co_applicant_age",
    "applicant_age_above_62",
    "debt_to_income_ratio",
]
cat_cols = [c for c in cat_cols if c in df.columns]
cat_df = df[cat_cols].copy()

high_card_cols = {"state_code", "derived_loan_product_type", "derived_dwelling_category"}
sentinel_values = {"8888", "9999", "8888.0", "9999.0"}
sentinel_cols = ["applicant_age", "co_applicant_age"]
for col in cat_df.columns:
    cat_df[col] = cat_df[col].astype(str).str.strip().str.lower()
    cat_df[col] = cat_df[col].replace({"nan": "missing", "": "missing"})
    if col in sentinel_cols:
        cat_df[col] = cat_df[col].replace(sentinel_values, "missing")
    if col in high_card_cols:
        cat_df[col] = cap_categories(cat_df[col], top_n=10, other_label="other")

cat_ohe = pd.get_dummies(cat_df, prefix=cat_df.columns, dtype=int)

num_bin_cols = {
    "loan_amount": 4,
    "income": 3,
    "interest_rate": 3,
    "combined_loan_to_value_ratio": 3,
    "loan_term": 5,
}

bin_df = pd.DataFrame(index=df.index)
for col, q in num_bin_cols.items():
    if col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce")
        if series.dropna().nunique() < 2:
            print(f"Lewati binning '{col}': variasi terlalu sedikit.")
            continue
        try:
            bin_df[f"{col}_bin"] = pd.qcut(series, q=q, duplicates="drop")
        except ValueError as exc:
            print(f"Lewati binning '{col}': {exc}")

if not bin_df.empty:
    bin_df = bin_df.astype("string").fillna("missing").astype(str)
    bin_ohe = pd.get_dummies(bin_df, prefix=bin_df.columns, dtype=int)
else:
    bin_ohe = pd.DataFrame(index=df.index)

item_df = pd.concat([binary_df, cat_ohe, bin_ohe], axis=1)
item_df = item_df.loc[:, item_df.sum(axis=0) > 0]

redundant_prefixes = ("income_bin_", "loan_amount_bin_")
redundant_cols = [c for c in item_df.columns if c.startswith(redundant_prefixes)]
if len(redundant_cols) > 0:
    item_df = item_df.drop(columns=redundant_cols)
    print("Kolom redundan dihapus:", len(redundant_cols))

if item_df.empty:
    raise ValueError("Itemset kosong. Cek kolom input atau kurangi filter.")

print("Jumlah item awal:", item_df.shape[1])

Kolom redundan dihapus: 8
Jumlah item awal: 142


In [32]:
import math

def _fd_bins(x):
    x = x.dropna()
    n = len(x)
    if n < 2:
        return None
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    if iqr <= 0:
        return None
    h = 2 * iqr * (n ** (-1 / 3))
    if h <= 0:
        return None
    bins = int(math.ceil((x.max() - x.min()) / h))
    return bins if bins > 0 else None

def _sturges_bins(n):
    return int(math.ceil(math.log2(n) + 1)) if n > 0 else None

def _sqrt_bins(n):
    return int(math.ceil(math.sqrt(n))) if n > 0 else None

def _evaluate_bins(x, bins, method="qcut"):
    if bins is None or bins < 2:
        return None
    try:
        if method == "qcut":
            binned = pd.qcut(x, q=bins, duplicates="drop")
        else:
            binned = pd.cut(x, bins=bins, duplicates="drop")
    except Exception:
        return None
    counts = binned.value_counts(dropna=True)
    if len(counts) == 0:
        return None
    min_count = int(counts.min())
    max_count = int(counts.max())
    mean_count = float(counts.mean())
    cv = float(counts.std(ddof=0) / mean_count) if mean_count > 0 else None
    empty_bins = int(bins - len(counts)) if method == "cut" else 0
    return {
        "method": method,
        "bins": int(bins),
        "min_count": min_count,
        "max_count": max_count,
        "cv": cv,
        "empty_bins": empty_bins,
    }

def _pick_best(results, n):
    if not results:
        return None
    df_res = pd.DataFrame(results).sort_values(
        by=["empty_bins", "cv", "bins"], ascending=[True, True, True]
    )
    min_ok = max(5, int(0.01 * n))
    df_res["min_ok"] = df_res["min_count"] >= min_ok
    best = df_res[df_res["min_ok"]].head(1)
    if best.empty:
        best = df_res.head(1)
    return best.iloc[0].to_dict()

def _outlier_share(x):
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return float(((x < lower) | (x > upper)).mean())

summary_rows = []
detail_results = {}

for col in num_bin_cols:
    if col not in df.columns:
        continue
    series = pd.to_numeric(df[col], errors="coerce").dropna()
    n = len(series)
    if n < 2:
        continue

    fd = _fd_bins(series)
    st = _sturges_bins(n)
    sq = _sqrt_bins(n)

    candidates = [3, 4, 5, fd, st, sq]
    candidates = sorted({c for c in candidates if c is not None and 2 <= c <= 10})

    results = []
    for c in candidates:
        res_q = _evaluate_bins(series, c, method="qcut")
        if res_q:
            results.append(res_q)
        res_c = _evaluate_bins(series, c, method="cut")
        if res_c:
            results.append(res_c)

    best = _pick_best(results, n)
    detail_results[col] = pd.DataFrame(results).sort_values(
        by=["empty_bins", "cv", "bins"], ascending=[True, True, True]
    )

    summary_rows.append(
        {
            "column": col,
            "n": n,
            "nunique": int(series.nunique()),
            "skew": float(series.skew()),
            "outlier_share": _outlier_share(series),
            "fd_bins": fd,
            "sturges_bins": st,
            "sqrt_bins": sq,
            "recommend_method": best["method"] if best else None,
            "recommend_bins": best["bins"] if best else None,
            "balance_cv": best["cv"] if best else None,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values(by=["column"])
display(summary_df)

print("Detail evaluasi tersedia di: detail_results['nama_kolom']")

,column,n,nunique,skew,outlier_share,fd_bins,sturges_bins,sqrt_bins,recommend_method,recommend_bins,balance_cv
3,combined_loan_to_value_ratio,65489,21841,208.175694,0.019331,19079.0,17,256,qcut,3,0.000022
1,income,88816,1338,275.613520,0.071046,269508.0,18,299,qcut,3,0.006714
2,interest_rate,63507,1812,0.661481,0.023730,151.0,17,253,qcut,3,0.029991
0,loan_amount,99994,454,52.572100,0.047583,8653.0,18,317,qcut,4,0.026352
4,loan_term,96491,296,-1.846206,0.000000,NaN,18,311,qcut,5,0.921725


Detail evaluasi tersedia di: detail_results['nama_kolom']


In [33]:
import numpy as np

def _bin_summary(binned, requested_bins):
    counts = binned.value_counts(dropna=True)
    if len(counts) == 0:
        return None
    mean_count = float(counts.mean())
    cv = float(counts.std(ddof=0) / mean_count) if mean_count > 0 else None
    actual_bins = int(len(counts))
    empty_bins = int(requested_bins - actual_bins) if requested_bins and requested_bins > actual_bins else 0
    return {
        "min_count": int(counts.min()),
        "max_count": int(counts.max()),
        "cv": cv,
        "actual_bins": actual_bins,
        "empty_bins": empty_bins,
    }

def _bin_quantile(x, bins):
    binned = pd.qcut(x, q=bins, duplicates="drop")
    return _bin_summary(binned, bins)

def _bin_winsor_cut(x, bins, lower=0.01, upper=0.99):
    x_clip = x.clip(x.quantile(lower), x.quantile(upper))
    binned = pd.cut(x_clip, bins=bins, duplicates="drop")
    return _bin_summary(binned, bins)

def _bin_log_cut(x, bins):
    min_val = float(x.min())
    shift = 0.0
    if min_val <= -1:
        shift = -min_val + 1
    x_log = np.log1p(x + shift)
    binned = pd.cut(x_log, bins=bins, duplicates="drop")
    return _bin_summary(binned, bins)

def _get_bins_for_col(col):
    if "summary_df" in globals():
        match = summary_df[summary_df["column"] == col]
        if not match.empty and pd.notna(match["recommend_bins"].iloc[0]):
            return int(match["recommend_bins"].iloc[0])
    return int(num_bin_cols.get(col, 4))

techniques = {
    "qcut": _bin_quantile,
    "winsor_cut": _bin_winsor_cut,
    "log_cut": _bin_log_cut,
}

rows = []
for col in num_bin_cols:
    if col not in df.columns:
        continue
    series = pd.to_numeric(df[col], errors="coerce").dropna()
    if series.nunique() < 2:
        continue
    bins = _get_bins_for_col(col)
    bins = max(2, min(10, bins))
    for name, func in techniques.items():
        stats = func(series, bins)
        if stats is None:
            continue
        rows.append({
            "column": col,
            "technique": name,
            "bins": bins,
            **stats,
        })

compare_df = pd.DataFrame(rows).sort_values(
    by=["column", "empty_bins", "cv", "bins"], ascending=[True, True, True, True]
).reset_index(drop=True)
display(compare_df)

best_df = compare_df.groupby("column", as_index=False).head(1).reset_index(drop=True)
print("Rekomendasi terbaik per kolom (empty_bins -> cv -> bins):")
display(best_df)

,column,technique,bins,min_count,max_count,cv,actual_bins,empty_bins
0,combined_loan_to_value_ratio,qcut,3,21829,21830,0.000022,3,0
1,combined_loan_to_value_ratio,winsor_cut,3,6992,38601,0.594446,3,0
2,combined_loan_to_value_ratio,log_cut,3,8,61680,1.292780,3,0
3,income,qcut,3,29348,29832,0.006714,3,0
4,income,log_cut,3,14,48104,0.714113,3,0
5,income,winsor_cut,3,1963,80979,1.228216,3,0
6,interest_rate,qcut,3,20290,21767,0.029991,3,0
7,interest_rate,winsor_cut,3,5353,31777,0.538467,3,0
8,interest_rate,log_cut,3,599,42660,0.811738,3,0
9,loan_amount,qcut,4,24147,25867,0.026352,4,0


Rekomendasi terbaik per kolom (empty_bins -> cv -> bins):


,column,technique,bins,min_count,max_count,cv,actual_bins,empty_bins
0,combined_loan_to_value_ratio,qcut,3,21829,21830,0.000022,3,0
1,income,qcut,3,29348,29832,0.006714,3,0
2,interest_rate,qcut,3,20290,21767,0.029991,3,0
3,loan_amount,qcut,4,24147,25867,0.026352,4,0
4,loan_term,winsor_cut,5,1299,72747,1.392366,5,0


**Perbandingan Visual: 3 Teknik Binning untuk Data Outlier**

Data dalam dataset ini memiliki outlier signifikan (income skew≈276, loan_amount skew≈53).
Tiga teknik binning yang paling cocok untuk data beroutlier dievaluasi di sini:

| Teknik | Ide Utama | Keunggulan vs Outlier |
|--------|-----------|----------------------|
| **qcut** | Tiap bin = jumlah baris sama | Outlier masuk bin ujung, bin lain tetap penuh |
| **winsor_cut** | Potong ekstrem 1-99%, lalu equal-width | Outlier ekstrem dikompresi, tidak dibuang |
| **log_cut** | Log transform → skala kompres → equal-width | Jarak ekstrem mengecil secara visual |

Metric perbandingan: **CV (Coefficient of Variation)** antar bin — makin rendah = distribusi bin makin merata.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

tech_colors_vis = {"qcut": "#1976D2", "winsor_cut": "#F57C00", "log_cut": "#388E3C"}
tech_pretty = {
    "qcut":       "1. Quantile Binning (qcut)",
    "winsor_cut": "2. Winsorization + Cut",
    "log_cut":    "3. Log Transform + Cut",
}

# ── Figure 1: Distribusi bin untuk income & loan_amount (kolom paling outlier) ──
show_cols = ["income", "loan_amount"]
fig1, axes = plt.subplots(len(show_cols), 4, figsize=(20, 4.8 * len(show_cols)))

for row_i, col in enumerate(show_cols):
    if col not in df.columns:
        continue
    series = pd.to_numeric(df[col], errors="coerce").dropna()
    bins_n = max(2, min(10, _get_bins_for_col(col)))
    outlier_pct = _outlier_share(series) * 100
    skew_val = float(series.skew())

    ax0 = axes[row_i, 0]
    ax0.hist(series.clip(series.quantile(0.005), series.quantile(0.995)),
             bins=40, color="#90A4AE", edgecolor="white", linewidth=0.4)
    ax0.set_title(f"{col.upper()}\nDistribusi Asli (clipped)", fontsize=9, fontweight="bold")
    ax0.set_xlabel("Nilai")
    ax0.set_ylabel("Frekuensi")
    ax0.text(0.97, 0.96, f"skew={skew_val:.1f}\noutlier={outlier_pct:.1f}%",
             transform=ax0.transAxes, ha="right", va="top", fontsize=7.5,
             bbox=dict(boxstyle="round,pad=0.25", fc="lightyellow", alpha=0.9))

    for col_i, tech in enumerate(["qcut", "winsor_cut", "log_cut"], start=1):
        ax = axes[row_i, col_i]
        try:
            if tech == "qcut":
                binned = pd.qcut(series, q=bins_n, duplicates="drop")
            elif tech == "winsor_cut":
                x2 = series.clip(series.quantile(0.01), series.quantile(0.99))
                binned = pd.cut(x2, bins=bins_n, duplicates="drop")
            else:
                shift = max(0.0, 1.0 - float(series.min()))
                x2 = np.log1p(series + shift)
                binned = pd.cut(x2, bins=bins_n, duplicates="drop")

            counts = binned.value_counts(dropna=True).sort_index()
            bars1 = ax.bar(range(len(counts)), counts.values,
                           color=tech_colors_vis[tech], edgecolor="white",
                           linewidth=0.5, alpha=0.9)
            for b_obj, cnt in zip(bars1, counts.values):
                ax.text(b_obj.get_x() + b_obj.get_width() / 2,
                        b_obj.get_height() * 1.01,
                        f"{cnt:,}", ha="center", va="bottom", fontsize=6.5)

            cv_row = compare_df[(compare_df["column"] == col) & (compare_df["technique"] == tech)]
            cv_str = f"CV = {float(cv_row['cv'].iloc[0]):.4f}" if len(cv_row) else ""
            ax.set_xticks(range(len(counts)))
            ax.set_xticklabels([f"Bin {j+1}" for j in range(len(counts))], fontsize=7.5)
            ax.set_title(f"{tech_pretty[tech]}\n{cv_str}", fontsize=8.5, fontweight="bold",
                         color=tech_colors_vis[tech])
            ax.set_xlabel("Bin")
            ax.set_ylabel("Jumlah Observasi")
        except Exception as e:
            ax.text(0.5, 0.5, str(e), transform=ax.transAxes, ha="center", fontsize=7)

fig1.suptitle(
    "Perbandingan 3 Teknik Binning untuk Data Outlier\n"
    "(Metric utama: CV — makin rendah = bin makin merata)",
    fontsize=11, fontweight="bold"
)
fig1.tight_layout(rect=[0, 0, 1, 0.96])
fig1.savefig("../reports/3-binning_per_column.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gambar 1 disimpan: ../reports/3-binning_per_column.png")


In [ ]:

# ── Figure 2: Heatmap CV + Rata-rata CV per Teknik ──
pivot_cv = compare_df.pivot_table(index="column", columns="technique", values="cv")
pivot_cv = pivot_cv.reindex(columns=["qcut", "winsor_cut", "log_cut"])

mean_cv_bar = compare_df.groupby("technique")["cv"].mean().reindex(["qcut", "winsor_cut", "log_cut"])
tech_colors_map = {"qcut": "#1976D2", "winsor_cut": "#F57C00", "log_cut": "#388E3C"}

fig2, (ax_heat, ax_bar) = plt.subplots(1, 2, figsize=(13, 5))

cmap = plt.get_cmap("RdYlGn_r")
vmax_val = float(compare_df["cv"].max())
im = ax_heat.imshow(pivot_cv.values, cmap=cmap, vmin=0, vmax=vmax_val, aspect="auto")
plt.colorbar(im, ax=ax_heat, label="CV")
ax_heat.set_xticks(range(len(pivot_cv.columns)))
ax_heat.set_xticklabels(pivot_cv.columns, fontsize=9)
ax_heat.set_yticks(range(len(pivot_cv.index)))
ax_heat.set_yticklabels(pivot_cv.index, fontsize=9)
ax_heat.set_title("Heatmap CV: Teknik vs Kolom\n(Merah = tidak merata, Hijau = merata)", fontsize=10, fontweight="bold")
for i in range(len(pivot_cv.index)):
    for j in range(len(pivot_cv.columns)):
        val = float(pivot_cv.values[i, j])
        if not np.isnan(val):
            text_color = "white" if val > vmax_val * 0.55 else "black"
            ax_heat.text(j, i, f"{val:.3f}", ha="center", va="center",
                        fontsize=8, fontweight="bold", color=text_color)

bar_col_list = [tech_colors_map.get(t, "#aaa") for t in mean_cv_bar.index]
bars2 = ax_bar.bar(mean_cv_bar.index, mean_cv_bar.values,
                   color=bar_col_list, edgecolor="white", linewidth=0.8)
ax_bar.set_title("Rata-rata CV per Teknik\n(Lebih rendah = distribusi lebih merata)", fontsize=10, fontweight="bold")
ax_bar.set_ylabel("Rata-rata CV")
ax_bar.set_xlabel("Teknik Binning")
for b, v in zip(bars2, mean_cv_bar.values):
    ax_bar.text(b.get_x() + b.get_width() / 2,
                b.get_height() + 0.008 * float(mean_cv_bar.max()),
                f"{v:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
winner_i = list(mean_cv_bar.index).index(mean_cv_bar.idxmin())
bars2[winner_i].set_edgecolor("#FFD700")
bars2[winner_i].set_linewidth(3)
ax_bar.text(winner_i,
            float(mean_cv_bar.iloc[winner_i]) + float(mean_cv_bar.max()) * 0.06,
            "★ TERBAIK", ha="center", fontsize=9, color="#B8860B", fontweight="bold")

fig2.tight_layout()
fig2.savefig("../reports/3-binning_cv_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gambar 2 disimpan: ../reports/3-binning_cv_comparison.png")


**Outline - Apriori & rule filtering**
- Import `mlxtend` and define thresholds (support, confidence, lift).
- Define correlated groups to remove tautological rules.
- Filter items by support and cap item count to control memory.
- Run Apriori and generate association rules.
- Relax confidence if rules are too few, then print final stats.

In [34]:
try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError as exc:
    raise ImportError("Paket mlxtend belum terpasang. Instal dulu: pip install mlxtend") from exc

MIN_SUPPORT = 0.001 
MIN_CONF = 0.6
MIN_LIFT = 1.5
MAX_ITEMSETS = 150 
MAX_LEN = 3

CORRELATED_GROUPS = [
    {"loan_type_", "derived_loan_product_type_"},
    {"lien_status_", "derived_loan_product_type_"},
    {"applicant_age_", "applicant_age_above_62_"},
    {"co_applicant_age_", "applicant_age_"},
    {"conforming_loan_limit_", "loan_size_"},
    {"preapproval_", "loan_purpose_"},
]

def is_tautological(row):
    items = set(row["antecedents"]).union(row["consequents"])
    for group in CORRELATED_GROUPS:
        prefixes_found = [p for p in group if any(str(item).startswith(p) for item in items)]
        if len(prefixes_found) >= 2:
            return True
    return False

def apply_filters(rules_df, min_lift):
    rules_df = rules_df[rules_df["lift"] >= min_lift]
    rules_df = rules_df[rules_df["consequents"].apply(lambda x: len(x) == 1)]
    tautology_mask = rules_df.apply(is_tautological, axis=1)
    tautology_count = int(tautology_mask.sum())
    rules_df = rules_df[~tautology_mask]
    return rules_df, tautology_count

item_support = item_df.mean(axis=0)
item_df_filtered = item_df.loc[:, item_support >= MIN_SUPPORT]

if item_df_filtered.shape[1] == 0:
    raise ValueError("Tidak ada item dengan support >= MIN_SUPPORT. Turunkan MIN_SUPPORT.")

# Batasi jumlah item agar Apriori tidak memakan memori berlebihan
if item_df_filtered.shape[1] > MAX_ITEMSETS:
    top_items = item_support.sort_values(ascending=False).head(MAX_ITEMSETS).index
    item_df_filtered = item_df_filtered[top_items]
    print("Item dibatasi ke:", len(top_items))

item_df_filtered = item_df_filtered.astype(bool)

print("Jumlah item setelah filter support:", item_df_filtered.shape[1])

frequent_itemsets = apriori(
    item_df_filtered,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_LEN,
    low_memory=True,
 )
print("Frequent itemsets:", len(frequent_itemsets))

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONF)
rules, tautology_removed = apply_filters(rules, MIN_LIFT)

if len(rules) < 10:
    relax_confs = [0.55, 0.5, 0.45]
    for conf in relax_confs:
        rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=conf)
        rules, tautology_removed = apply_filters(rules, MIN_LIFT)
        if len(rules) >= 10:
            MIN_CONF = conf
            break

print("Jumlah rules setelah filter:", len(rules))
print("Rules terhapus (tautologi):", tautology_removed)
print("Threshold akhir -> support:", MIN_SUPPORT, "confidence:", MIN_CONF, "lift:", MIN_LIFT)

Jumlah item setelah filter support: 131
Frequent itemsets: 107600
Jumlah rules setelah filter: 6106
Rules terhapus (tautologi): 2479
Threshold akhir -> support: 0.001 confidence: 0.6 lift: 1.5


**Outline - Reporting & export**
- Normalize/clean item names for consistent grouping.
- Select diverse top rules with group caps and lift threshold.
- Build display tables for the report and show top 10.
- Save full rules to CSV and a text summary to reports.

In [35]:
def format_itemset(itemset):
    return ", ".join(sorted([str(x) for x in itemset]))
# 2. Binning teknik yang cocok untuk data outlier

def parse_itemset_str(text):
    if not isinstance(text, str) or text.strip() == "":
        return []
    return [t.strip() for t in text.split(",")]

MINOR_PREFIXES = ("total_units_", "occupancy_type_", "derived_race_")

def canonical_item(item):
    text = str(item).strip().lower()
    if text == "lien_status_2":
        return "subordinate_lien"
    if "subordinate lien" in text:
        return "subordinate_lien"
    return text

def canonicalize_items(items):
    return [canonical_item(i) for i in items]

def core_antecedent_key(items):
    canon = canonicalize_items(items)
    core = [i for i in canon if not i.startswith(MINOR_PREFIXES)]
    return tuple(sorted(core))

def rule_signature(antecedent_items, consequent_items):
    ant_key = core_antecedent_key(antecedent_items)
    cons_key = tuple(sorted(canonicalize_items(consequent_items)))
    return (ant_key, cons_key)

CONSEQUENT_GROUPS = [
    {"name": "loan_product", "prefixes": {"loan_type_", "derived_loan_product_type_", "lien_status_"}},
    {"name": "income_group", "prefixes": {"income_bracket_", "income_bin_"}},
    {"name": "loan_size_group", "prefixes": {"loan_size_", "loan_amount_bin_", "conforming_loan_limit_"}},
    {"name": "preapproval_purpose", "prefixes": {"preapproval_", "loan_purpose_"}},
    {"name": "age_group", "prefixes": {"applicant_age_", "co_applicant_age_", "applicant_age_above_62_"}},
    {"name": "dwelling_construction", "prefixes": {"construction_method_", "derived_dwelling_category_"}},
]

REPORT_MIN_LIFT = 2.0

def get_consequent_key(consequent):
    text = str(consequent).strip().lower()
    for group in CONSEQUENT_GROUPS:
        if any(text.startswith(p) for p in group["prefixes"]):
            return group["name"]
    return f"cons_{text}"

def init_state():
    return {"selected": [], "used_signatures": set(), "used_ants": set(), "group_counts": {}}

def add_rules(base_df, caps, state, top_n, allow_over_cap=False):
    for _, row in base_df.iterrows():
        if len(state["selected"]) >= top_n:
            break
        ant_items = row.get("antecedent_items")
        cons_items = row.get("consequent_items")
        if not isinstance(ant_items, list):
            ant_items = parse_itemset_str(row.get("antecedents", ""))
        if not isinstance(cons_items, list):
            cons_items = parse_itemset_str(row.get("consequents", ""))
        ant_key = core_antecedent_key(ant_items)
        if ant_key in state["used_ants"]:
            continue
        signature = rule_signature(ant_items, cons_items)
        if signature in state["used_signatures"]:
            continue
        cons_key = get_consequent_key(cons_items[0] if cons_items else "")
        if caps is not None:
            cap = caps.get(cons_key, 1)
            if not allow_over_cap and state["group_counts"].get(cons_key, 0) >= cap:
                continue
        state["selected"].append(row)
        state["used_signatures"].add(signature)
        state["used_ants"].add(ant_key)
        state["group_counts"][cons_key] = state["group_counts"].get(cons_key, 0) + 1
    return state

def select_report_rules(df_rules, top_n=10):
    base = df_rules[df_rules["lift"] >= REPORT_MIN_LIFT].copy()
    if base.empty:
        return pd.DataFrame(), REPORT_MIN_LIFT
    base = base.sort_values(by=["lift", "confidence", "support"], ascending=False)
    state = init_state()
    strict_caps = {
        "dwelling_construction": 1,
        "loan_product": 2,
        "loan_size_group": 2,
        "income_group": 2,
    }
    state = add_rules(base, strict_caps, state, top_n)
    if len(state["selected"]) < top_n:
        relaxed_caps = dict(strict_caps)
        relaxed_caps["loan_product"] = 3
        relaxed_caps["loan_size_group"] = 3
        relaxed_caps["income_group"] = 3
        state = add_rules(base, relaxed_caps, state, top_n)
    if len(state["selected"]) < top_n:
        state = add_rules(base, {}, state, top_n, allow_over_cap=True)
    return pd.DataFrame(state["selected"]), REPORT_MIN_LIFT

def safe_write_csv(df, path):
    try:
        df.to_csv(path, index=False)
        return path
    except PermissionError:
        alt_path = path.replace(".csv", "_new.csv")
        df.to_csv(alt_path, index=False)
        return alt_path

def safe_write_txt(lines, path):
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        return path
    except PermissionError:
        alt_path = path.replace(".txt", "_new.txt")
        with open(alt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        return alt_path

rules_table = rules.copy()
rules_table["antecedent_items"] = rules_table["antecedents"].apply(lambda x: sorted([str(i) for i in x]))
rules_table["consequent_items"] = rules_table["consequents"].apply(lambda x: sorted([str(i) for i in x]))
rules_table["antecedents"] = rules_table["antecedent_items"].apply(lambda items: ", ".join(items))
rules_table["consequents"] = rules_table["consequent_items"].apply(lambda items: ", ".join(items))

rules_table = rules_table[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
    "antecedent_items",
    "consequent_items",
]]
rules_table = rules_table.sort_values(by=["lift", "confidence", "support"], ascending=False)

rules_for_selection = rules_table.copy()
rules_table = rules_table[["antecedents", "consequents", "support", "confidence", "lift"]]

top_rules, used_min_lift = select_report_rules(rules_for_selection, top_n=10)
top_rules_display = top_rules[["antecedents", "consequents", "support", "confidence", "lift"]] if not top_rules.empty else top_rules
print(f"Top 10 aturan untuk laporan (lift >= {used_min_lift}):")
display(top_rules_display)

rules_csv_path = "../reports/3-association-rules.csv"
rules_txt_path = "../reports/3-association-rules.txt"

csv_path_used = safe_write_csv(rules_table, rules_csv_path)

if len(top_rules_display) < 10:
    print("Peringatan: aturan untuk laporan kurang dari 10. Pertimbangkan menambah item atau melonggarkan filter.")

lines = []
lines.append("Ringkasan Aturan Asosiasi (Top 10)")
lines.append("")
for _, row in top_rules_display.iterrows():
    lines.append(
        f"- Jika {row['antecedents']} maka {row['consequents']} "
        f"(support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.2f})"
    )

txt_path_used = safe_write_txt(lines, rules_txt_path)

print("Tabel rules disimpan ke:", csv_path_used)
print("Ringkasan disimpan ke:", txt_path_used)

Top 10 aturan untuk laporan (lift >= 2.0):


,antecedents,consequents,support,confidence,lift
27838,derived_dwelling_category_multifamily:site-bui...,total_units_5-24,0.00265,0.718157,251.969857
71214,"conforming_loan_limit_missing, construction_me...",derived_dwelling_category_multifamily:site-built,0.00413,1.000000,242.116223
1092,derived_dwelling_category_multifamily:site-built,conforming_loan_limit_missing,0.00413,1.000000,235.280000
71213,"construction_method_1, derived_dwelling_catego...",conforming_loan_limit_missing,0.00413,1.000000,235.280000
31443,"loan_term_bin_missing, purchaser_type_2",reverse_mortgage_1,0.00231,0.942857,137.836341
51458,"loan_purpose_4, reverse_mortgage_1111",debt_to_income_ratio_exempt,0.00147,1.000000,55.306416
84939,"co_applicant_age_45-54, debt_to_income_ratio_e...",reverse_mortgage_1111,0.00176,1.000000,55.184327
1490,reverse_mortgage_1,loan_term_bin_missing,0.00684,1.000000,28.545247
1096,conforming_loan_limit_missing,occupancy_type_3,0.00425,1.000000,13.505402
30728,"conforming_loan_limit_nc, purchaser_type_2",loan_type_3,0.00126,0.940299,12.526540


Tabel rules disimpan ke: ../reports/3-association-rules.csv
Ringkasan disimpan ke: ../reports/3-association-rules.txt


In [ ]:

# ── Kesimpulan Binning & Append ke Report ──
mean_cv_summary = compare_df.groupby("technique")["cv"].mean().reindex(["qcut", "winsor_cut", "log_cut"])
income_info = summary_df[summary_df["column"] == "income"].iloc[0]
loan_info = summary_df[summary_df["column"] == "loan_amount"].iloc[0]

binning_section = "\n".join([
    "",
    "=" * 65,
    "ANALISIS TEKNIK BINNING UNTUK DATA OUTLIER",
    "=" * 65,
    "",
    "Karakteristik Data Numerik (kolom paling terpengaruh outlier):",
    f"  - income      : skew={income_info['skew']:.1f}, outlier_share={income_info['outlier_share']*100:.1f}%",
    f"  - loan_amount : skew={loan_info['skew']:.1f}, outlier_share={loan_info['outlier_share']*100:.1f}%",
    "",
    "3 Teknik Binning Terbaik untuk Data Outlier:",
    "",
    "1. Quantile Binning (qcut)",
    "   Prinsip : Setiap bin berisi jumlah observasi yang SAMA.",
    "   Outlier : Otomatis masuk ke bin paling kiri/kanan -- tidak dibuang.",
    f"   CV rata2: {mean_cv_summary['qcut']:.4f}  <-- TERKECIL = paling seimbang",
    "   Cocok   : Ideal untuk Apriori karena min_support tiap bin terpenuhi.",
    "",
    "2. Winsorization + Equal-width Cut (winsor_cut)",
    "   Prinsip : Potong nilai ekstrem di percentile 1-99%, lalu bagi lebar sama.",
    "   Outlier : Ditekan ke batas percentile (tidak dibuang, hanya dikompres).",
    f"   CV rata2: {mean_cv_summary['winsor_cut']:.4f}  <-- sedang",
    "   Cocok   : Saat batas bin harus intuitif untuk interpretasi bisnis.",
    "",
    "3. Log Transform + Equal-width Cut (log_cut)",
    "   Prinsip : Transformasi log memperkecil skewness, lalu bagi lebar sama.",
    "   Outlier : Dikompres ke skala log sehingga jarak visual mengecil.",
    f"   CV rata2: {mean_cv_summary['log_cut']:.4f}  <-- tertinggi",
    "   Cocok   : Data yang benar-benar berdistribusi log-normal.",
    "",
    "KESIMPULAN:",
    "  Teknik terbaik untuk data outlier = QUANTILE BINNING (qcut)",
    f"  CV rata-rata qcut = {mean_cv_summary['qcut']:.4f} vs",
    f"  winsor_cut = {mean_cv_summary['winsor_cut']:.4f}, log_cut = {mean_cv_summary['log_cut']:.4f}",
    "  qcut menghasilkan distribusi bin paling merata karena bekerja",
    "  pada rank/posisi data, bukan pada nilai absolut -- sehingga",
    "  outlier ekstrem tidak menyebabkan bin kosong atau sangat kecil.",
    "  Winsorization = alternatif baik jika batas bin harus mudah dibaca.",
    "  Log Transform tidak direkomendasikan: CV tetap tinggi karena",
    "  distribusi tidak seluruhnya log-normal.",
    "  --> qcut digunakan sebagai teknik utama pada pipeline Apriori ini.",
    "",
    "Gambar analisis:",
    "  - 3-binning_per_column.png   : distribusi bin per kolom (income, loan_amount)",
    "  - 3-binning_cv_comparison.png: heatmap CV + ranking rata-rata per teknik",
])

with open("../reports/3-association-rules.txt", "a", encoding="utf-8") as f:
    f.write(binning_section + "\n")

print("Analisis binning ditambahkan ke: ../reports/3-association-rules.txt")
print(binning_section)
